# Notebook 07 — Cost & latency report
**Goal:** Turn LangSmith's automatically-captured traces into a presentation-ready cost and latency summary — no new instrumentation needed.

Tracing has been on since notebook 01 (`LANGCHAIN_TRACING_V2=true`), so every ingestion, chat turn, RAG query, and compliance check run through this project has already been recorded in the `copilot-mvp` LangSmith project. This notebook just reads and summarizes it.

By the end of this notebook you will have:
- Pulled every traced run and computed total GPT-4o-mini cost and token usage
- Latency distribution (avg / p50 / p95 / max) for LLM calls
- Cost and latency broken down by operation type — e.g. "what does one full agent chat turn cost and how long does it take?"
- Tool-level latency (`query_corpus`, `check_compliance`, `ingest_video`)
- Saved everything to `data/metrics/` and regenerated `data/metrics/SUMMARY.md`

**Important caveat:** embedding calls (`text-embedding-3-small`) go through the raw OpenAI client in `embedder.py`, not a LangChain-wrapped call, so they are **not traced**. Every cost number below is GPT-4o-mini generation only. Embeddings are priced low enough (~$0.02 / 1M tokens) that this gap is real but small — worth stating rather than ignoring.

**Quick reference:** for a fast look at past reports without re-running anything, open `data/metrics/SUMMARY.md`.

## Step 1 — Environment check

In [ ]:
import sys
sys.path.append('..')

from src.utils.config import LANGCHAIN_API_KEY, LANGCHAIN_PROJECT

print('✅ LangSmith key loaded:', LANGCHAIN_API_KEY[:10] + '...')
print(f'✅ Project: {LANGCHAIN_PROJECT}')

## Step 2 — Pull traces and compute the report

In [ ]:
from src.utils.langsmith_report import generate_cost_latency_report

report = generate_cost_latency_report()

if 'error' in report:
    print(f'⚠️ {report["error"]}')
else:
    totals = report['totals']
    print(f'Trace window: {report["trace_window"]["earliest"]} → {report["trace_window"]["latest"]}')
    print(f'Total runs traced: {totals["total_runs"]}')
    print(f'  LLM calls: {totals["llm_calls"]} | Tool calls: {totals["tool_calls"]} | Top-level operations: {totals["root_operations"]}')
    print(f'  Model(s): {", ".join(totals["models_used"])}')
    print()
    print(f'💰 Total GPT cost: ${totals["total_cost_usd"]:.4f}')
    print(f'   Tokens: {totals["total_prompt_tokens"]:,} prompt + {totals["total_completion_tokens"]:,} completion')

## Step 3 — LLM call latency distribution

In [ ]:
lat = report['llm_call_latency_seconds']
print('=== LLM CALL LATENCY (seconds) ===')
print(f'  n:   {lat["n"]}')
print(f'  avg: {lat["avg"]}s')
print(f'  p50: {lat["p50"]}s')
print(f'  p95: {lat["p95"]}s')
print(f'  max: {lat["max"]}s')

## Step 4 — Cost & latency per operation type
Attributes every LLM call's cost back to its top-level operation — e.g. one
`AgentExecutor` run is one full chat turn from the Streamlit UI (agent reasoning
+ tool calls + final answer). `RunnableSequence` covers RAG chain calls made
directly in notebooks (02, 04) without the agent's tool-selection overhead.

In [ ]:
print('=== COST & LATENCY PER OPERATION TYPE ===\n')
print(f'{"Operation":<20}{"N":>4}{"Avg cost":>12}{"Total cost":>13}{"Avg latency":>13}{"p95 latency":>13}')
for name, stats in report['operation_breakdown'].items():
    print(
        f'{name:<20}{stats["n"]:>4}'
        f'${stats["avg_cost_usd"]:>10.6f} '
        f'${stats["total_cost_usd"]:>11.6f} '
        f'{stats["latency"]["avg"]:>11.2f}s '
        f'{stats["latency"]["p95"]:>11.2f}s'
    )

## Step 5 — Tool-level latency
How long each agent tool takes end-to-end (includes its own internal retrieval/LLM calls).

In [ ]:
print('=== TOOL-LEVEL LATENCY (seconds) ===\n')
print(f'{"Tool":<20}{"N":>4}{"Avg":>8}{"p50":>8}{"p95":>8}{"Max":>8}')
for name, stats in report['tool_latency_seconds'].items():
    print(f'{name:<20}{stats["n"]:>4}{stats["avg"]:>8.2f}{stats["p50"]:>8.2f}{stats["p95"]:>8.2f}{stats["max"]:>8.2f}')

## Step 6 — Save and regenerate the one-place summary

In [ ]:
from src.utils.langsmith_report import save_report, generate_summary_md

latest_path, archive_path = save_report(report)
print(f'✅ Saved latest report to {latest_path}')
print(f'✅ Archived this run to {archive_path}')

summary_path = generate_summary_md()
print(f'✅ Regenerated {summary_path}')

## Notes

**What this does and doesn't cover:** GPT-4o-mini generation cost only — embedding
cost is not traced (see caveat at the top). Also doesn't include Pinecone's own
hosting cost, which is separate from LangSmith/OpenAI and billed by Pinecone directly
(free tier covers this project's current scale).

**Re-running:** Safe any time — pulls whatever's in the LangSmith project at that
moment. `langsmith_report.json` always reflects the latest run; every run is archived
under `data/metrics/runs/<timestamp>.json` and indexed in `data/metrics/SUMMARY.md`,
so cost/latency trends are trackable over time as more videos get ingested and more
questions get asked.

**Interpreting operation-type costs:** `AgentExecutor` (full chat turns via the
Streamlit UI) costs more per call than `RunnableSequence` (direct RAG chain calls)
because the agent does extra reasoning to decide which tool(s) to call — that's
expected overhead, not a bug.